In [1]:
!pip install torch torchvision torchaudio --quiet

In [2]:
import os
import csv
import math
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

In [3]:
DATA_PATH    = "/content/ceemdan_components.csv"   # adjust if needed
OUTPUT_CSV   = "/content/informer_experiment_results.csv"
OUTPUT_TXT   = "/content/informer_experiment_summary.txt"

# Component lists
IMF_COLS      = ["IMF1", "IMF2", "IMF3", "IMF4", "IMF5", "IMF6", "IMF7", "IMF8"]
RESIDUAL_COL  = "Residual"
TARGET_COL    = "Close"
# IMF8 uses first-diff + StandardScaler; all others use MinMaxScaler
MINMAX_COMPS  = ["IMF1", "IMF2", "IMF3", "IMF4", "IMF5", "IMF6", "IMF7", "Residual"]
ALL_COMPONENTS = IMF_COLS + [RESIDUAL_COL]   # 9 total

# Informer hyperparameters
INF_SEQ_LEN    = 96
INF_LABEL_LEN  = 48
INF_PRED_LEN   = 1
INF_D_MODEL    = 256
INF_N_HEADS    = 8
INF_ENC_LAYERS = 2
INF_DEC_LAYERS = 1
INF_D_FF       = INF_D_MODEL * 4
INF_DROPOUT    = 0.05
INF_BATCH      = 32
INF_LR         = 1e-4
INF_EPOCHS     = 50
INF_PATIENCE   = 7

# Experiment settings
N_EXPERIMENTS      = 50         # seeds 1 → 50
ROLLING_VOL_WINDOW = 30
MULTISTEP_HORIZONS = [5, 21]

# Checkpoint file — written after every seed for crash recovery
CHECKPOINT_CSV = OUTPUT_CSV     # reuses the same output path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
def set_seed(seed: int):
    """Fully deterministic seeding."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [5]:
def create_informer_windows(series: np.ndarray,
                             seq_len: int, label_len: int, pred_len: int):
    """
    Sliding-window for Informer (pred_len=1, no leakage).
    Encoder : series[i : i+seq_len]
    Decoder : series[i+seq_len-label_len : i+seq_len] + zeros(pred_len)
    Target  : series[i+seq_len]
    Returns enc_x (N, seq_len, 1), dec_x (N, label_len+pred_len, 1), y (N,)
    """
    enc_x, dec_x, targets = [], [], []
    for i in range(len(series) - seq_len - pred_len + 1):
        enc_seq    = series[i: i + seq_len]
        label_part = series[i + seq_len - label_len: i + seq_len]
        pred_part  = np.zeros(pred_len, dtype=np.float32)
        dec_seq    = np.concatenate([label_part, pred_part])
        target_val = series[i + seq_len]
        enc_x.append(enc_seq)
        dec_x.append(dec_seq)
        targets.append(target_val)
    enc_x   = np.array(enc_x,   dtype=np.float32)[..., np.newaxis]
    dec_x   = np.array(dec_x,   dtype=np.float32)[..., np.newaxis]
    targets = np.array(targets, dtype=np.float32)
    return enc_x, dec_x, targets


In [6]:
class InformerDataset(Dataset):
    def __init__(self, enc_x, dec_x, targets):
        self.enc_x   = torch.from_numpy(enc_x)
        self.dec_x   = torch.from_numpy(dec_x)
        self.targets = torch.from_numpy(targets)
    def __len__(self): return len(self.targets)
    def __getitem__(self, idx):
        return self.enc_x[idx], self.dec_x[idx], self.targets[idx]

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ---- ProbSparse Attention ----
class ProbSparseAttention(nn.Module):
    """
    Simplified ProbSparse Self-Attention (Informer paper).
    Falls back to full scaled dot-product when sequence is short.
    """
    def __init__(self, d_model, n_heads, factor=5, attention_dropout=0.05):
        super().__init__()
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.factor   = factor
        self.dropout  = nn.Dropout(attention_dropout)
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, queries, keys, values, attn_mask=None):
        B, L_Q, _ = queries.shape
        L_K       = keys.shape[1]
        Q = self.q_proj(queries).view(B, L_Q, self.n_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(keys).view(B, L_K, self.n_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(values).view(B, L_K, self.n_heads, self.d_head).transpose(1, 2)
        scale  = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out  = torch.matmul(attn, V)
        out  = out.transpose(1, 2).contiguous().view(B, L_Q, -1)
        return self.out_proj(out)


# ---- Encoder / Decoder Layers ----
class InformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()
        self.attn  = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.drop(self.attn(x, x, x)))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x


class InformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.05):
        super().__init__()
        self.self_attn  = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.cross_attn = ProbSparseAttention(d_model, n_heads, attention_dropout=dropout)
        self.ff         = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, enc_out):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x)))
        x = self.norm2(x + self.drop(self.cross_attn(x, enc_out, enc_out)))
        x = self.norm3(x + self.drop(self.ff(x)))
        return x


# ---- Full Informer ----
class Informer(nn.Module):
    """
    Informer for single-step forecasting.
    Input  enc_x : (B, seq_len, 1)
    Input  dec_x : (B, label_len+pred_len, 1)
    Output       : (B,)  — one scalar per sample
    """
    def __init__(self,
                 d_model    = INF_D_MODEL,
                 n_heads    = INF_N_HEADS,
                 enc_layers = INF_ENC_LAYERS,
                 dec_layers = INF_DEC_LAYERS,
                 d_ff       = INF_D_FF,
                 dropout    = INF_DROPOUT,
                 seq_len    = INF_SEQ_LEN,
                 label_len  = INF_LABEL_LEN,
                 pred_len   = INF_PRED_LEN):
        super().__init__()
        self.pred_len  = pred_len
        self.label_len = label_len
        self.enc_embed = nn.Linear(1, d_model)
        self.dec_embed = nn.Linear(1, d_model)
        self.enc_pos   = PositionalEncoding(d_model, max_len=seq_len + 10,            dropout=dropout)
        self.dec_pos   = PositionalEncoding(d_model, max_len=label_len + pred_len + 10, dropout=dropout)
        self.encoder   = nn.ModuleList([
            InformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(enc_layers)])
        self.decoder   = nn.ModuleList([
            InformerDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(dec_layers)])
        self.enc_norm  = nn.LayerNorm(d_model)
        self.dec_norm  = nn.LayerNorm(d_model)
        self.proj      = nn.Linear(d_model, 1)

    def forward(self, enc_x, dec_x):
        enc_out = self.enc_pos(self.enc_embed(enc_x))
        for layer in self.encoder:
            enc_out = layer(enc_out)
        enc_out = self.enc_norm(enc_out)
        dec_out = self.dec_pos(self.dec_embed(dec_x))
        for layer in self.decoder:
            dec_out = layer(dec_out, enc_out)
        dec_out = self.dec_norm(dec_out)
        out = self.proj(dec_out[:, -self.pred_len:, :])   # (B, pred_len, 1)
        return out.squeeze(-1).squeeze(-1)                 # (B,)


In [8]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """MAE, RMSE, MAPE, R² on original-scale arrays."""
    y_true = np.array(y_true, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask = np.abs(y_true) > 1e-8
    mape = float(np.mean(np.abs(
        (y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

In [9]:
def _train_informer(train_sc: np.ndarray,
                    val_sc  : np.ndarray,
                    test_sc : np.ndarray,
                    patience: int = INF_PATIENCE) -> np.ndarray:
    """
    Train a fresh Informer on pre-scaled sequences.
    Returns test_predictions_scaled (1-D numpy array).
    """
    # Build windows  (prepend context for val/test)
    val_context  = np.concatenate([train_sc[-INF_SEQ_LEN:], val_sc])
    test_context = np.concatenate([val_sc[-INF_SEQ_LEN:],   test_sc])

    enc_tr, dec_tr, y_tr = create_informer_windows(
        train_sc,    INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)
    enc_vl, dec_vl, y_vl = create_informer_windows(
        val_context, INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)
    enc_te, dec_te, y_te = create_informer_windows(
        test_context,INF_SEQ_LEN, INF_LABEL_LEN, INF_PRED_LEN)

    if len(enc_tr) == 0:
        raise ValueError("Training set too short for Informer windows.")

    pin = (DEVICE.type == "cuda")
    tr_dl = DataLoader(InformerDataset(enc_tr, dec_tr, y_tr),
                       INF_BATCH, shuffle=False, pin_memory=pin)
    vl_dl = DataLoader(InformerDataset(enc_vl, dec_vl, y_vl),
                       INF_BATCH, shuffle=False)
    te_ds  = InformerDataset(enc_te, dec_te, y_te)

    model = Informer().to(DEVICE)
    opt   = Adam(model.parameters(), lr=INF_LR)
    crit  = nn.MSELoss()
    best_v = float("inf");  best_w = None;  no_imp = 0

    for epoch in range(1, INF_EPOCHS + 1):
        model.train()
        for eb, db, yb in tr_dl:
            eb, db, yb = eb.to(DEVICE), db.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(eb, db), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vl_losses = []
        with torch.no_grad():
            for eb, db, yb in vl_dl:
                vl_losses.append(
                    crit(model(eb.to(DEVICE), db.to(DEVICE)),
                         yb.to(DEVICE)).item())
        vl_loss = float(np.mean(vl_losses))

        if vl_loss < best_v:
            best_v = vl_loss
            best_w = {k: v.clone().cpu() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_w.items()})
    model.eval()
    te_dl = DataLoader(te_ds, INF_BATCH, shuffle=False)
    preds = []
    with torch.no_grad():
        for eb, db, _ in te_dl:
            preds.append(model(eb.to(DEVICE), db.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)   # scaled predictions

In [10]:
def train_minmax_component(comp, df_tr, df_vl, df_te, scalers) -> np.ndarray:
    """Train Informer with MinMaxScaler. Returns original-scale test preds."""
    sc     = scalers[comp]
    tr_sc  = sc.transform(df_tr[[comp]].values).flatten().astype(np.float32)
    vl_sc  = sc.transform(df_vl[[comp]].values).flatten().astype(np.float32)
    te_sc  = sc.transform(df_te[[comp]].values).flatten().astype(np.float32)
    preds_sc = _train_informer(tr_sc, vl_sc, te_sc)
    return sc.inverse_transform(
        preds_sc.reshape(-1, 1)).flatten().astype(np.float64)


def train_imf8_firstdiff(df_tr, df_vl, df_te) -> np.ndarray:
    """
    IMF8 first-difference + StandardScaler pipeline.
    1. Δ[t] = IMF8[t] - IMF8[t-1]
    2. Fit StandardScaler on train Δ only.
    3. Train Informer on scaled Δ.
    4. Inverse-scale → Δ_hat.
    5. Reconstruct: IMF8_hat[t] = IMF8_val[-1] + cumsum(Δ_hat).
    Returns original-scale test predictions.
    """
    tr_raw = df_tr["IMF8"].values.astype(np.float64)
    vl_raw = df_vl["IMF8"].values.astype(np.float64)
    te_raw = df_te["IMF8"].values.astype(np.float64)

    d_tr = np.diff(tr_raw)
    d_vl = np.diff(np.concatenate([[tr_raw[-1]], vl_raw]))
    d_te = np.diff(np.concatenate([[vl_raw[-1]], te_raw]))

    ss = StandardScaler()
    ss.fit(d_tr.reshape(-1, 1))
    d_tr_sc = ss.transform(d_tr.reshape(-1, 1)).flatten().astype(np.float32)
    d_vl_sc = ss.transform(d_vl.reshape(-1, 1)).flatten().astype(np.float32)
    d_te_sc = ss.transform(d_te.reshape(-1, 1)).flatten().astype(np.float32)

    delta_preds_sc = _train_informer(d_tr_sc, d_vl_sc, d_te_sc)
    delta_preds = ss.inverse_transform(
        delta_preds_sc.reshape(-1, 1)).flatten().astype(np.float64)

    anchor = float(vl_raw[-1])
    return anchor + np.cumsum(delta_preds)


In [11]:
def run_experiment(df_train, df_val, df_test, seed: int) -> tuple:
    """
    Full CEEMDAN-Informer pipeline for one seed:
      1. set_seed (deterministic).
      2. Fit MinMaxScalers on train (8 components).
      3. Train 9 Informers (IMF8 via first-diff, rest via MinMax).
      4. Reconstruct: ŷ = Σ all 9 component predictions.
      5. Compute & return metrics + artefacts.
    """
    set_seed(seed)
    t0 = time.time()

    # Fit MinMaxScalers on train only
    scalers = {}
    for comp in MINMAX_COMPS:
        sc = MinMaxScaler(feature_range=(0, 1))
        sc.fit(df_train[[comp]].values)
        scalers[comp] = sc

    comp_preds = {}

    # Train Informer for each MinMax component
    for comp in MINMAX_COMPS:
        comp_preds[comp] = train_minmax_component(
            comp, df_train, df_val, df_test, scalers)
        print(f"    [{comp}] done", flush=True)

    # Train IMF8 (first-diff pipeline)
    comp_preds["IMF8"] = train_imf8_firstdiff(df_train, df_val, df_test)
    print(f"    [IMF8] done (first-diff)", flush=True)

    t_elapsed = time.time() - t0

    # Reconstruct: ŷ = Σ all components
    n_test = len(df_test)
    y_hat  = np.zeros(n_test, dtype=np.float64)
    for comp in ALL_COMPONENTS:
        y_hat += comp_preds[comp][:n_test]

    y_true  = df_test[TARGET_COL].values.astype(np.float64)
    min_len = min(len(y_hat), len(y_true))
    y_hat_a  = y_hat[:min_len]
    y_true_a = y_true[:min_len]

    metrics = compute_metrics(y_true_a, y_hat_a)
    metrics["training_time_sec"] = t_elapsed
    return metrics, comp_preds, scalers, y_true_a, y_hat_a

In [12]:
_CKPT_FIELDS = ["experiment", "seed", "MAE", "RMSE", "MAPE", "R2",
                "training_time_sec",
                "multi_5d_MAE", "multi_5d_RMSE",
                "multi_21d_MAE", "multi_21d_RMSE"]


def _load_checkpoint() -> tuple:
    """
    Read CHECKPOINT_CSV (if it exists) and return:
        completed_seeds : set of int  — seeds already finished
        all_results     : list of dict — one dict per completed seed
    """
    if not os.path.exists(CHECKPOINT_CSV):
        return set(), []
    try:
        ck = pd.read_csv(CHECKPOINT_CSV)
        if ck.empty:
            return set(), []
        records = ck.to_dict(orient="records")
        seeds   = {int(r["seed"]) for r in records}
        print(f"  ↻  Checkpoint found: {len(seeds)} seeds already done "
              f"({sorted(seeds)[0]}–{sorted(seeds)[-1]}).  Resuming…")
        return seeds, records
    except Exception as e:
        print(f"  ⚠  Could not read checkpoint ({e}). Starting fresh.")
        return set(), []


def _append_checkpoint(row: dict):
    """
    Append a single result row to CHECKPOINT_CSV immediately after each seed.
    Creates the file with headers on first write.
    """
    write_header = not os.path.exists(CHECKPOINT_CSV)
    with open(CHECKPOINT_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=_CKPT_FIELDS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)


def cell13_load_and_run():
    """Load dataset, split chronologically, run 50 experiments with checkpointing."""
    print(f"Device  : {DEVICE}")
    print(f"PyTorch : {torch.__version__}")
    print(f"Running : {N_EXPERIMENTS} experiments  (seeds 1 → {N_EXPERIMENTS})")
    print("=" * 65)

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    train_mask    = df["Date"].dt.year <= 2022
    test_mask     = df["Date"].dt.year >= 2023
    df_train_full = df[train_mask].reset_index(drop=True)
    df_test       = df[test_mask].reset_index(drop=True)
    val_size      = int(len(df_train_full) * 0.10)
    df_train      = df_train_full.iloc[:-val_size].reset_index(drop=True)
    df_val        = df_train_full.iloc[-val_size:].reset_index(drop=True)

    print(f"Train : {len(df_train):>5}  "
          f"({df_train.Date.min().date()} → {df_train.Date.max().date()})")
    print(f"Val   : {len(df_val):>5}  "
          f"({df_val.Date.min().date()} → {df_val.Date.max().date()})")
    print(f"Test  : {len(df_test):>5}  "
          f"({df_test.Date.min().date()} → {df_test.Date.max().date()})")
    print(f"\nIMF8 range: train max={df_train['IMF8'].max():.0f}  "
          f"test max={df_test['IMF8'].max():.0f}  ← first-diff fix applied")
    print("=" * 65)

    # ── Checkpoint recovery ──────────────────────────────────────────────────
    completed_seeds, all_results = _load_checkpoint()
    # Reconstruct _last_* from checkpoint rows so plots still work after resume
    _last_y_true = _last_y_hat = _last_comp_preds = None

    for seed in range(1, N_EXPERIMENTS + 1):

        # ── Skip already-completed seeds ──────────────────────────────────────
        if seed in completed_seeds:
            print(f"[{seed:02d}/{N_EXPERIMENTS}] seed={seed}  ✓ skipped (checkpoint)",
                  flush=True)
            continue

        print(f"\n[{seed:02d}/{N_EXPERIMENTS}] seed={seed}", flush=True)
        m, cp, scalers, yt, yp = run_experiment(
            df_train, df_val, df_test, seed=seed)
        m["experiment"] = seed
        m["seed"]       = seed
        all_results.append(m)

        # ── Persist result immediately so a crash loses at most 1 seed ────────
        _append_checkpoint(m)

        print(f"  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  "
              f"MAPE={m['MAPE']:.4f}%  R²={m['R2']:.4f}  "
              f"time={m['training_time_sec']:.1f}s", flush=True)
        _last_y_true     = yt
        _last_y_hat      = yp
        _last_comp_preds = cp

    # ── Sort results by seed for consistent summary output ───────────────────
    all_results.sort(key=lambda r: r["seed"])

    print("\n✅ All 50 experiments complete.")
    return df, df_test, all_results, _last_y_true, _last_y_hat


In [13]:
def cell14_save_csv(all_results):
    """
    Re-write the full CSV in seed order.
    (Individual rows were already appended live during cell13; this step
    ensures the final file is clean and sorted even after a resumed run.)
    """
    all_results_sorted = sorted(all_results, key=lambda r: r["seed"])
    with open(OUTPUT_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=_CKPT_FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(all_results_sorted)
    print(f"Per-experiment CSV → {OUTPUT_CSV}")
    return pd.read_csv(OUTPUT_CSV)

In [14]:
def cell15_summary(all_results):
    """Print and save the 50-run aggregate statistics."""
    maes  = [r["MAE"]               for r in all_results]
    rmses = [r["RMSE"]              for r in all_results]
    mapes = [r["MAPE"]              for r in all_results]
    r2s   = [r["R2"]                for r in all_results]
    times = [r["training_time_sec"] for r in all_results]

    sep   = "=" * 72
    lines = [
        sep,
        "  CEEMDAN-Informer  —  50-RUN STATISTICAL SUMMARY",
        sep,
        f'  {"Metric":<14} {"Mean":>12} {"± Std":>12} {"Min":>10} {"Max":>10}',
        "-" * 72,
    ]
    for label, arr in [("MAE", maes), ("RMSE", rmses),
                        ("MAPE (%)", mapes), ("R²", r2s), ("Time (s)", times)]:
        mn, sd, mi, mx = (np.mean(arr), np.std(arr),
                          np.min(arr), np.max(arr))
        lines.append(f"  {label:<14} {mn:>12.4f} {sd:>12.4f}"
                     f" {mi:>10.4f} {mx:>10.4f}")
    mn_t, sd_t = np.mean(times), np.std(times)
    lines += [
        "-" * 72,
        f'  {"Time (min)":<14} {mn_t/60:>12.4f} {sd_t/60:>12.4f}',
        sep,
    ]
    txt = "\n".join(lines)
    print("\n" + txt)
    with open(OUTPUT_TXT, "w") as f:
        f.write(txt + "\n")
    print(f"\nSummary → {OUTPUT_TXT}")
    return maes, rmses, mapes, r2s, times

In [15]:
def cell16_plot_actual_vs_predicted(df_test, y_true, y_hat):
    """Two-panel chart: price overlay + residual bar."""
    test_dates = df_test["Date"].values[:len(y_true)]

    fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                              gridspec_kw={"height_ratios": [3, 1]})
    ax = axes[0]
    ax.plot(test_dates, y_true, color="#1565C0", lw=1.8, label="Actual Close")
    ax.plot(test_dates, y_hat,  color="#E53935", lw=1.3,
            ls="--", label="Predicted Close")
    ax.set_title("CEEMDAN-Informer: Actual vs Predicted (Test 2023–2025)",
                 fontsize=14, fontweight="bold")
    ax.set_ylabel("NIFTY 50 Close")
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    errs = y_true - y_hat
    ax2.bar(test_dates, errs, color="#7B1FA2", alpha=0.6, width=1)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_title("Prediction Error (Actual − Predicted)")
    ax2.set_xlabel("Date"); ax2.set_ylabel("Error")
    ax2.grid(True, alpha=0.3)

    m50 = compute_metrics(y_true, y_hat)
    fig.text(0.01, 0.97,
             f'Seed 50  |  MAE={m50["MAE"]:.2f}  RMSE={m50["RMSE"]:.2f}  '
             f'MAPE={m50["MAPE"]:.4f}%  R²={m50["R2"]:.4f}',
             fontsize=10, va="top")
    plt.tight_layout()
    out1 = "/content/informer_actual_vs_predicted.png"
    plt.savefig(out1, dpi=150, bbox_inches="tight"); plt.close()
    print(f"Plot saved → {out1}")


In [16]:
def cell17_metric_distributions(maes, rmses, mapes, r2s):
    """Four-panel histogram of MAE, RMSE, MAPE, R² across 50 seeds."""
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    data_pairs = [
        ("MAE",      maes,  "#1565C0"),
        ("RMSE",     rmses, "#C62828"),
        ("MAPE (%)", mapes, "#2E7D32"),
        ("R²",       r2s,   "#F57F17"),
    ]
    for ax, (label, arr, col) in zip(axes, data_pairs):
        ax.hist(arr, bins=15, color=col, alpha=0.8, edgecolor="white")
        ax.axvline(np.mean(arr), color="black", ls="--", lw=1.5,
                   label=f"Mean={np.mean(arr):.3f}")
        ax.set_title(label, fontweight="bold")
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.suptitle("Metric Distributions across 50 Independent Runs (CEEMDAN-Informer)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out2 = "/content/informer_metric_distributions.png"
    plt.savefig(out2, dpi=150, bbox_inches="tight"); plt.close()
    print(f"Plot saved → {out2}")



In [17]:
def cell18_volatility_regime(df_test, y_true, y_hat):
    """Split test period into high / low volatility regimes and report metrics."""
    close_all = df_test[TARGET_COL].values.astype(np.float64)
    log_ret   = np.log(close_all[1:] / close_all[:-1])
    rol_vol   = pd.Series(log_ret).rolling(ROLLING_VOL_WINDOW).std()
    vol_vals  = rol_vol.values[:len(y_true)]
    vol_med   = float(np.nanmedian(vol_vals))
    vol_vals  = np.where(np.isnan(vol_vals), vol_med, vol_vals)

    high_mask = vol_vals >= vol_med
    low_mask  = ~high_mask
    m_h = compute_metrics(y_true[high_mask], y_hat[high_mask])
    m_l = compute_metrics(y_true[low_mask],  y_hat[low_mask])

    print("\n" + "=" * 66)
    print("  VOLATILITY REGIME VALIDATION (seed=50)")
    print(f"  30-day rolling vol | Median = {vol_med:.6f}")
    print("=" * 66)
    print(f'  {"Metric":<12} {"High Volatility":>18} {"Low Volatility":>18}')
    print("-" * 66)
    for k in ["MAE", "RMSE", "MAPE", "R2"]:
        print(f"  {k:<12} {m_h[k]:>18.4f} {m_l[k]:>18.4f}")
    print("=" * 66)
    print(f"  High-vol days : {high_mask.sum()}")
    print(f"  Low-vol  days : {low_mask.sum()}")


In [18]:
def main():
    df, df_test, all_results, y_true, y_hat = cell13_load_and_run()
    cell14_save_csv(all_results)
    maes, rmses, mapes, r2s, times         = cell15_summary(all_results)
    cell16_plot_actual_vs_predicted(df_test, y_true, y_hat)
    cell17_metric_distributions(maes, rmses, mapes, r2s)
    cell18_volatility_regime(df_test, y_true, y_hat)

if __name__ == "__main__":
    main()

Device  : cuda
PyTorch : 2.10.0+cu128
Running : 50 experiments  (seeds 1 → 50)
Train :  1770  (2015-01-09 → 2022-03-16)
Val   :   196  (2022-03-17 → 2022-12-30)
Test  :   633  (2023-01-02 → 2025-07-25)

IMF8 range: train max=16783  test max=23158  ← first-diff fix applied

[01/50] seed=1
    [IMF1] done
    [IMF2] done
    [IMF3] done
    [IMF4] done
    [IMF5] done
    [IMF6] done
    [IMF7] done
    [Residual] done
    [IMF8] done (first-diff)
  MAE=2756.16  RMSE=2999.86  MAPE=12.2516%  R²=-0.3346  time=211.5s

[02/50] seed=2
    [IMF1] done
    [IMF2] done
    [IMF3] done
    [IMF4] done
    [IMF5] done
    [IMF6] done
    [IMF7] done
    [Residual] done
    [IMF8] done (first-diff)
  MAE=2473.91  RMSE=2888.38  MAPE=10.7721%  R²=-0.2373  time=237.5s

[03/50] seed=3
    [IMF1] done
    [IMF2] done
    [IMF3] done
    [IMF4] done
    [IMF5] done
    [IMF6] done
    [IMF7] done
    [Residual] done
    [IMF8] done (first-diff)
  MAE=235.85  RMSE=299.09  MAPE=1.0598%  R²=0.9867  time=212

IndexError: boolean index did not match indexed array along axis 0; size of axis is 633 but size of corresponding boolean axis is 632